In [1]:
import os
from pathlib import Path

print("Searching for videos...\n")

VIDEO_ROOT = None
source_folders = {}

for root, dirs, files in os.walk("/kaggle/input"):
    mp4s = [f for f in files if f.endswith('.mp4')]
    if mp4s:
        folder = Path(root)
        folder_name = folder.name
        source_folders[folder_name] = {"path": folder, "count": len(mp4s)}
        print(f"  {folder_name}: {len(mp4s)} videos at {folder}")

total = sum(v["count"] for v in source_folders.values())
print(f"\nTotal videos found: {total}")

Searching for videos...

  gemini_omni_flash: 32 videos at /kaggle/input/datasets/shantanuvedanteog/all-corpus-videos/all_videos/gemini_omni_flash
  kling: 32 videos at /kaggle/input/datasets/shantanuvedanteog/all-corpus-videos/all_videos/kling
  pexels: 64 videos at /kaggle/input/datasets/shantanuvedanteog/all-corpus-videos/all_videos/pexels
  wan: 40 videos at /kaggle/input/datasets/shantanuvedanteog/all-corpus-videos/all_videos/wan
  ltx: 40 videos at /kaggle/input/datasets/shantanuvedanteog/all-corpus-videos/all_videos/ltx
  hunyuan: 40 videos at /kaggle/input/datasets/shantanuvedanteog/all-corpus-videos/all_videos/hunyuan
  seedance: 32 videos at /kaggle/input/datasets/shantanuvedanteog/all-corpus-videos/all_videos/seedance

Total videos found: 280


In [2]:
import cv2

def extract_frames(video_path, output_folder, num_frames=6):
    """
    Extract N evenly-spaced frames from a video.
    Skips very first and last frames to avoid fade-in/out artefacts.
    Saves as 95% quality JPEG.
    """
    output_folder.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    
    if not cap.isOpened():
        return False, "Could not open video"
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    if total_frames < num_frames + 2:
        cap.release()
        return False, f"Too few frames ({total_frames})"
    
    # Skip first and last frame
    step = (total_frames - 2) / (num_frames - 1)
    frame_indices = [int(1 + i * step) for i in range(num_frames)]
    
    stem = video_path.stem
    extracted = 0
    
    for i, idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            t = idx / fps if fps > 0 else 0
            out_path = output_folder / f"{stem}_frame{i+1:02d}_t{t:.2f}s.jpg"
            cv2.imwrite(str(out_path), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            extracted += 1
    
    cap.release()
    return extracted == num_frames, f"Extracted {extracted}/{num_frames}"

print("Frame extraction function ready.")
print("Settings: 6 frames per video, JPEG 95%, skip first/last frame")

Frame extraction function ready.
Settings: 6 frames per video, JPEG 95%, skip first/last frame


In [3]:
FRAMES_ROOT = Path("/kaggle/working/frames")
FRAMES_ROOT.mkdir(exist_ok=True)

results = {"success": 0, "failed": 0, "failures": []}

for source_name, info in sorted(source_folders.items()):
    source_path = info["path"]
    frame_output = FRAMES_ROOT / source_name
    
    videos = sorted(source_path.glob("*.mp4"))
    print(f"\n=== {source_name}: {len(videos)} videos ===")
    
    for i, video in enumerate(videos):
        success, msg = extract_frames(video, frame_output)
        if success:
            results["success"] += 1
        else:
            results["failed"] += 1
            results["failures"].append(f"{source_name}/{video.name}: {msg}")
        
        if (i+1) % 10 == 0:
            print(f"  Processed {i+1}/{len(videos)}")
    
    # Verify frame count for this source
    frames_created = len(list(frame_output.glob("*.jpg")))
    expected = len(videos) * 6
    status = "OK" if frames_created == expected else f"MISMATCH (expected {expected})"
    print(f"  Frames: {frames_created} {status}")

print(f"\n{'='*60}")
print(f"EXTRACTION COMPLETE")
print(f"Success: {results['success']}")
print(f"Failed: {results['failed']}")

if results["failures"]:
    print(f"\nFailures:")
    for f in results["failures"]:
        print(f"  {f}")


=== gemini_omni_flash: 32 videos ===
  Processed 10/32
  Processed 20/32
  Processed 30/32
  Frames: 192 OK

=== hunyuan: 40 videos ===
  Processed 10/40
  Processed 20/40
  Processed 30/40
  Processed 40/40
  Frames: 240 OK

=== kling: 32 videos ===
  Processed 10/32
  Processed 20/32
  Processed 30/32
  Frames: 192 OK

=== ltx: 40 videos ===
  Processed 10/40
  Processed 20/40
  Processed 30/40
  Processed 40/40
  Frames: 240 OK

=== pexels: 64 videos ===
  Processed 10/64
  Processed 20/64
  Processed 30/64
  Processed 40/64
  Processed 50/64
  Processed 60/64
  Frames: 384 OK

=== seedance: 32 videos ===
  Processed 10/32
  Processed 20/32
  Processed 30/32
  Frames: 192 OK

=== wan: 40 videos ===
  Processed 10/40
  Processed 20/40
  Processed 30/40
  Processed 40/40
  Frames: 240 OK

EXTRACTION COMPLETE
Success: 280
Failed: 0


In [4]:
print("Frame count verification:\n")

total_frames = 0
for source_name in sorted(source_folders.keys()):
    frame_dir = FRAMES_ROOT / source_name
    if frame_dir.exists():
        frame_count = len(list(frame_dir.glob("*.jpg")))
        video_count = source_folders[source_name]["count"]
        expected = video_count * 6
        status = "PASS" if frame_count == expected else "FAIL"
        print(f"  {source_name}: {frame_count} frames from {video_count} videos [{status}]")
        total_frames += frame_count

print(f"\nTotal frames: {total_frames}")
print(f"Expected: {sum(v['count'] for v in source_folders.values()) * 6}")

Frame count verification:

  gemini_omni_flash: 192 frames from 32 videos [PASS]
  hunyuan: 240 frames from 40 videos [PASS]
  kling: 192 frames from 32 videos [PASS]
  ltx: 240 frames from 40 videos [PASS]
  pexels: 384 frames from 64 videos [PASS]
  seedance: 192 frames from 32 videos [PASS]
  wan: 240 frames from 40 videos [PASS]

Total frames: 1680
Expected: 1680


In [5]:
print("Sample frames (one per source):\n")
for source_name in sorted(source_folders.keys()):
    frame_dir = FRAMES_ROOT / source_name
    if frame_dir.exists():
        sample = sorted(frame_dir.glob("*.jpg"))[0]
        size_kb = sample.stat().st_size / 1024
        print(f"  {source_name}: {sample.name} ({size_kb:.1f} KB)")

Sample frames (one per source):

  gemini_omni_flash: w2_001_gemini_omni_flash_frame01_t0.04s.jpg (71.8 KB)
  hunyuan: w2_001_hunyuan_20260719_132500_frame01_t0.04s.jpg (51.7 KB)
  kling: w2_001_kling_frame01_t0.04s.jpg (50.0 KB)
  ltx: w2_001_ltx_20260719_105436_frame01_t0.04s.jpg (70.7 KB)
  pexels: pexels_animal_18106001_frame01_t0.04s.jpg (63.9 KB)
  seedance: w2_001_seedance_frame01_t0.04s.jpg (38.0 KB)
  wan: w2_001_wan_20260719_162444_frame01_t0.06s.jpg (92.9 KB)


In [6]:
import shutil

shutil.make_archive("/kaggle/working/all_frames", "zip", str(FRAMES_ROOT))

for source_name in sorted(source_folders.keys()):
    frame_dir = FRAMES_ROOT / source_name
    if frame_dir.exists():
        shutil.make_archive(f"/kaggle/working/frames_{source_name}", "zip", str(frame_dir))

!ls -lh /kaggle/working/*.zip
print("\nDownload all_frames.zip from the Output panel.")
print("Or download per-source ZIPs individually.")

-rw-r--r-- 1 root root 146M Jul 23 18:50 /kaggle/working/all_frames.zip
-rw-r--r-- 1 root root  18M Jul 23 18:50 /kaggle/working/frames_gemini_omni_flash.zip
-rw-r--r-- 1 root root  14M Jul 23 18:50 /kaggle/working/frames_hunyuan.zip
-rw-r--r-- 1 root root  17M Jul 23 18:50 /kaggle/working/frames_kling.zip
-rw-r--r-- 1 root root  17M Jul 23 18:50 /kaggle/working/frames_ltx.zip
-rw-r--r-- 1 root root  37M Jul 23 18:50 /kaggle/working/frames_pexels.zip
-rw-r--r-- 1 root root  17M Jul 23 18:50 /kaggle/working/frames_seedance.zip
-rw-r--r-- 1 root root  28M Jul 23 18:50 /kaggle/working/frames_wan.zip

Download all_frames.zip from the Output panel.
Or download per-source ZIPs individually.
